In [1]:

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import pathlib

# ---- Paths & data ----
DATA_DIR = "/home/jovyan/.cache/kagglehub/datasets/obulisainaren/multi-cancer/versions/3/Multi Cancer/Multi Cancer/Breast Cancer"
IMG_SIZE = (256, 256)   
BATCH_SIZE = 8
SEED = 1337

data_dir = pathlib.Path(DATA_DIR)

# ===== Dataset split: train (70%), val (15%), test (15%) =====
train_ds = keras.utils.image_dataset_from_directory(
    data_dir, labels="inferred", label_mode="int",
    image_size=IMG_SIZE, batch_size=BATCH_SIZE,
    validation_split=0.15, subset="training", seed=SEED, shuffle=True
)

valtest_ds = keras.utils.image_dataset_from_directory(
    data_dir, labels="inferred", label_mode="int",
    image_size=IMG_SIZE, batch_size=BATCH_SIZE,
    validation_split=0.15, subset="validation", seed=SEED, shuffle=False
)

# Split val/test 50/50 from the validation subset
val_batches = int(0.5 * tf.data.experimental.cardinality(valtest_ds).numpy())
val_ds = valtest_ds.take(val_batches)
test_ds = valtest_ds.skip(val_batches)

class_names = train_ds.class_names
num_classes = len(class_names)
print("Classes:", class_names)

# ===== Prefetch for performance (same as second snippet) =====
AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_ds.shuffle(1000).prefetch(AUTOTUNE)
val_ds   = val_ds.prefetch(AUTOTUNE)
test_ds  = test_ds.prefetch(AUTOTUNE)

# ===== Data augmentation (same as second snippet) =====
data_augmentation = keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.05),
    layers.RandomZoom(0.1),
])

# ===== Normalization (same as second snippet) =====
# NOTE: We use simple Rescaling(1./255) for ALL models, including transfer learning.
# This matches the second snippet; it's fine for your experiments.
def normalize_block(x):
    return layers.Rescaling(1./255)(x)

# =========================================
# Classifier head (shared for TL models)
# =========================================
def classifier_head(x, num_classes, dropout=0.3):
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dropout(dropout)(x)
    return layers.Dense(num_classes, activation="softmax")(x)

# =========================================
# 1) Baseline CNN (from scratch)
#   - keep architecture flexible, but preprocessing matches second snippet
# =========================================
def build_baseline_cnn(input_shape, num_classes):
    inputs = keras.Input(shape=input_shape)
    x = data_augmentation(inputs)
    x = normalize_block(x)

    # Three conv blocks, two convs per block (as in your first snippet's baseline)
    for filters in [32, 64, 128]:
        x = layers.Conv2D(filters, 3, padding="same", activation="relu")(x)
        x = layers.Conv2D(filters, 3, padding="same", activation="relu")(x)
        x = layers.MaxPooling2D()(x)
        x = layers.Dropout(0.25)(x)

    x = layers.Flatten()(x)
    x = layers.Dense(256, activation="relu")(x)
    x = layers.Dropout(0.5)(x)
    outputs = layers.Dense(num_classes, activation="softmax")(x)
    return keras.Model(inputs, outputs, name="baseline_cnn")

# =========================================
# 2) ResNet50 (prebuilt)
# =========================================
from tensorflow.keras.applications import ResNet50
def build_resnet50(input_shape, num_classes, weights="imagenet", train_base=False):
    inputs = keras.Input(shape=input_shape)
    x = data_augmentation(inputs)
    x = normalize_block(x)
    base = ResNet50(include_top=False, weights=weights, input_tensor=x)
    base.trainable = train_base
    outputs = classifier_head(base.output, num_classes, dropout=0.3)
    return keras.Model(inputs, outputs, name="resnet50")

# =========================================
# 3) VGG16 (prebuilt)
# =========================================
from tensorflow.keras.applications import VGG16
def build_vgg16(input_shape, num_classes, weights="imagenet", train_base=False):
    inputs = keras.Input(shape=input_shape)
    x = data_augmentation(inputs)
    x = normalize_block(x)
    base = VGG16(include_top=False, weights=weights, input_tensor=x)
    base.trainable = train_base
    outputs = classifier_head(base.output, num_classes, dropout=0.3)
    return keras.Model(inputs, outputs, name="vgg16")

# =========================================
# 4) InceptionV3 (prebuilt)
# =========================================
from tensorflow.keras.applications import InceptionV3
def build_inceptionv3(input_shape, num_classes, weights="imagenet", train_base=False):
    inputs = keras.Input(shape=input_shape)
    x = data_augmentation(inputs)
    x = normalize_block(x)
    base = InceptionV3(include_top=False, weights=weights, input_tensor=x)
    base.trainable = train_base
    outputs = classifier_head(base.output, num_classes, dropout=0.3)
    return keras.Model(inputs, outputs, name="inceptionv3")

# =========================================
# 5) EfficientNetB0 (prebuilt)
# =========================================
from tensorflow.keras.applications import EfficientNetB0
def build_efficientnetb0(input_shape, num_classes, weights="imagenet", train_base=False):
    inputs = keras.Input(shape=input_shape)
    x = data_augmentation(inputs)
    x = normalize_block(x)
    base = EfficientNetB0(include_top=False, weights=weights, input_tensor=x)
    base.trainable = train_base
    outputs = classifier_head(base.output, num_classes, dropout=0.3)
    return keras.Model(inputs, outputs, name="efficientnetb0")

# =========================================
# Pick a model to train (same interface)
# =========================================
MODEL_NAME = "cnn"   # one of: "cnn", "resnet", "vgg", "inception", "efficientnet"
WEIGHTS = "imagenet"          # "imagenet" or None
FINE_TUNE_BASE = False        # set True to unfreeze backbone later

input_shape = (*IMG_SIZE, 3)

if MODEL_NAME == "cnn":
    model = build_baseline_cnn(input_shape, num_classes)
elif MODEL_NAME == "resnet":
    model = build_resnet50(input_shape, num_classes, weights=WEIGHTS, train_base=FINE_TUNE_BASE)
elif MODEL_NAME == "vgg":
    model = build_vgg16(input_shape, num_classes, weights=WEIGHTS, train_base=FINE_TUNE_BASE)
elif MODEL_NAME == "inception":
    model = build_inceptionv3(input_shape, num_classes, weights=WEIGHTS, train_base=FINE_TUNE_BASE)
elif MODEL_NAME == "efficientnet":
    model = build_efficientnetb0(input_shape, num_classes, weights=WEIGHTS, train_base=FINE_TUNE_BASE)
else:
    raise ValueError("Unknown MODEL_NAME")
'''
#Summarize the model and get layers and parameters
def summarize_model(m):
    total = m.count_params()
    trainable = np.sum([w.numpy().size for w in m.trainable_weights]) if tf.executing_eagerly() else m.count_params()
    non_trainable = total - trainable
    head_dense = m.layers[-1]  # final Dense(num_classes)
    head_params = head_dense.count_params()  # = (features * num_classes) + num_classes
    feat_dim = (head_params // num_classes) - 1 if num_classes > 0 else None
    print(f"\n== {m.name} ==")
    print(f"Layers: {len(m.layers)}")
    print(f"Params: total={total:,} | trainable={trainable:,} | non-trainable={non_trainable:,}")
    print(f"Head (Dense) params: {head_params:,}  -> feature_dim feeding head ≈ {feat_dim}")
    try:
        m.summary(line_length=120, expand_nested=True, show_trainable=True)
    except:
        pass

import numpy as np

# Build all variants with your current num_classes and input size (no training here)
models_to_check = {
    "cnn":         build_baseline_cnn(input_shape, num_classes),
    "resnet50":    build_resnet50(input_shape, num_classes, weights=WEIGHTS, train_base=FINE_TUNE_BASE),
    "vgg16":       build_vgg16(input_shape, num_classes, weights=WEIGHTS, train_base=FINE_TUNE_BASE),
    "inceptionv3": build_inceptionv3(input_shape, num_classes, weights=WEIGHTS, train_base=FINE_TUNE_BASE),
    "efficientnetb0": build_efficientnetb0(input_shape, num_classes, weights=WEIGHTS, train_base=FINE_TUNE_BASE),
}

for name, m in models_to_check.items():
    summarize_model(m)

'''
# =========================================
# Compile & Train
# =========================================
model.compile(
    optimizer=keras.optimizers.Adam(1e-3),
    loss=keras.losses.SparseCategoricalCrossentropy(),
    metrics=["accuracy"]
)

callbacks = [
    keras.callbacks.EarlyStopping(patience=4, restore_best_weights=True, monitor="val_accuracy"),
]

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=20,
    callbacks=callbacks
)

# ===== Evaluate on the held-out TEST set (matches second snippet practice) =====
test_loss, test_acc = model.evaluate(test_ds, verbose=0)
print(f"{model.name}  |  Test acc: {test_acc:.4f}  |  Test loss: {test_loss:.4f}")

# =========================
# Quick sanity checks (drop-in)
# =========================
import numpy as np
print("\n--- Sanity checks on TEST evaluation ---")

# 0) Cardinalities (did we evaluate the full test set?)
card_train = tf.data.experimental.cardinality(train_ds).numpy()
card_val   = tf.data.experimental.cardinality(val_ds).numpy()
card_test  = tf.data.experimental.cardinality(test_ds).numpy()
print(f"Batches -> train:{card_train}  val:{card_val}  test:{card_test}")

# 1) Class order consistency
print("Class order (train):", class_names)
try:
    print("Class order (test):", test_ds.class_names)  # may not exist on tf.data
except AttributeError:
    print("Class order (test): uses same pipeline; relying on 'class_names' from train")

# 2) Collect y_true from the EXACT test_ds you passed to evaluate()
y_true = np.concatenate([y.numpy() for _, y in test_ds], axis=0)
print("Test size (images):", y_true.shape[0])

# 3) Get predictions on the SAME test_ds and compute accuracy manually
y_prob = model.predict(test_ds, verbose=0)
y_pred = y_prob.argmax(axis=1)
manual_acc = (y_pred == y_true).mean()
print(f"Manual accuracy: {manual_acc:.4f}  |  Keras evaluate accuracy: {test_acc:.4f}")

# 4) Per-class counts (catch degenerate splits)
from collections import Counter
counts = Counter(y_true.tolist())
print("Per-class counts in TEST:", dict(counts))

# 5) Confusion matrix & report
try:
    from sklearn.metrics import confusion_matrix, classification_report
    cm = confusion_matrix(y_true, y_pred, labels=range(num_classes))
    print("Confusion matrix (rows=true, cols=pred):\n", cm)
    print(classification_report(y_true, y_pred, digits=4))
except Exception as e:
    # Fallback without sklearn
    cm_tf = tf.math.confusion_matrix(y_true, y_pred, num_classes=num_classes).numpy()
    print("Confusion matrix (rows=true, cols=pred) [tf fallback]:\n", cm_tf)
    # Simple per-class precision/recall (macro-ish) without sklearn
    tp = np.diag(cm_tf)
    precision = tp / np.maximum(1, cm_tf.sum(axis=0))
    recall    = tp / np.maximum(1, cm_tf.sum(axis=1))
    print("Per-class precision (fallback):", np.round(precision, 4))
    print("Per-class recall (fallback):",    np.round(recall, 4))

# 6) Sanity: ensure we didn't accidentally evaluate on val_ds
val_size  = sum([y.shape[0] for _, y in val_ds])
print(f"Val size (images): {val_size}")
assert y_true.shape[0] != val_size or manual_acc != 1.0 or test_acc != 1.0, \
    "Warning: test and val sizes match; double-check you didn't evaluate on val_ds by mistake."

2025-09-30 22:08:15.892194: I tensorflow/core/util/port.cc:110] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-09-30 22:08:15.949525: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2025-09-30 22:08:17.018783: W tensorflow/compiler/tf2tensorrt/utils/py_utils.cc:38] TF-TRT Warning: Could not find TensorRT


Found 10000 files belonging to 2 classes.
Using 8500 files for training.


2025-09-30 22:08:21.112942: I tensorflow/core/common_runtime/gpu/gpu_device.cc:1639] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 9559 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 2080 Ti, pci bus id: 0000:60:00.0, compute capability: 7.5
2025-09-30 22:08:21.114426: I tensorflow/core/common_runtime/gpu/gpu_device.cc:1639] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 9559 MB memory:  -> device: 1, name: NVIDIA GeForce RTX 2080 Ti, pci bus id: 0000:61:00.0, compute capability: 7.5
2025-09-30 22:08:21.115851: I tensorflow/core/common_runtime/gpu/gpu_device.cc:1639] Created device /job:localhost/replica:0/task:0/device:GPU:2 with 9559 MB memory:  -> device: 2, name: NVIDIA GeForce RTX 2080 Ti, pci bus id: 0000:b2:00.0, compute capability: 7.5


Found 10000 files belonging to 2 classes.
Using 1500 files for validation.
Classes: ['breast_benign', 'breast_malignant']

== baseline_cnn ==
Layers: 19
Params: total=33,842,210 | trainable=33,842,210 | non-trainable=0
Head (Dense) params: 514  -> feature_dim feeding head ≈ 256
Model: "baseline_cnn"
___________________________________________________________________________________________________________________________________
 Layer (type)                                         Output Shape                                    Param #           Trainable  
 input_2 (InputLayer)                                 [(None, 256, 256, 3)]                           0                 Y          
                                                                                                                                   
 sequential (Sequential)                              (None, 256, 256, 3)                             0                 N          
|¯¯¯¯¯¯¯¯¯¯¯¯¯¯¯¯¯¯¯¯¯¯¯¯¯¯¯¯¯¯¯¯¯¯¯¯¯¯

2025-09-30 22:08:45.758993: E tensorflow/core/grappler/optimizers/meta_optimizer.cc:954] layout failed: INVALID_ARGUMENT: Size of values 0 does not match size of permutation 4 @ fanin shape inbaseline_cnn/dropout/dropout/SelectV2-2-TransposeNHWCToNCHW-LayoutOptimizer
2025-09-30 22:08:55.985693: I tensorflow/core/kernels/data/shuffle_dataset_op.cc:422] Filling up shuffle buffer (this may take a while): 383 of 1000
2025-09-30 22:09:05.986787: I tensorflow/core/kernels/data/shuffle_dataset_op.cc:422] Filling up shuffle buffer (this may take a while): 817 of 1000
2025-09-30 22:09:13.144622: I tensorflow/core/kernels/data/shuffle_dataset_op.cc:450] Shuffle buffer filled.
2025-09-30 22:09:13.694915: I tensorflow/compiler/xla/stream_executor/cuda/cuda_dnn.cc:432] Loaded cuDNN version 8600
2025-09-30 22:09:14.267864: I tensorflow/compiler/xla/service/service.cc:168] XLA service 0x55a0c8b593d0 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
2025-09-30 22:

1063/1063 [==============================] - 72s 37ms/step - loss: 0.7033 - accuracy: 0.4964 - val_loss: 0.6979 - val_accuracy: 0.0000e+00
Epoch 2/20
1063/1063 [==============================] - 42s 34ms/step - loss: 0.6933 - accuracy: 0.4965 - val_loss: 0.6828 - val_accuracy: 1.0000
Epoch 3/20
1063/1063 [==============================] - 42s 35ms/step - loss: 0.6933 - accuracy: 0.5029 - val_loss: 0.6967 - val_accuracy: 0.0000e+00
Epoch 4/20
1063/1063 [==============================] - 41s 33ms/step - loss: 0.6933 - accuracy: 0.4952 - val_loss: 0.7004 - val_accuracy: 0.0000e+00
Epoch 5/20
1063/1063 [==============================] - 39s 33ms/step - loss: 0.6933 - accuracy: 0.4949 - val_loss: 0.6942 - val_accuracy: 0.0000e+00
Epoch 6/20
1063/1063 [==============================] - 40s 33ms/step - loss: 0.6933 - accuracy: 0.4994 - val_loss: 0.6946 - val_accuracy: 0.0000e+00
baseline_cnn  |  Test acc: 1.0000  |  Test loss: 0.6828

--- Sanity checks on TEST evaluation ---
Batches -> train:

In [2]:

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import pathlib

# ---- Paths & data ----
DATA_DIR = "/home/jovyan/.cache/kagglehub/datasets/obulisainaren/multi-cancer/versions/3/Multi Cancer/Multi Cancer/Breast Cancer"
IMG_SIZE = (256, 256)   
BATCH_SIZE = 8
SEED = 1337

data_dir = pathlib.Path(DATA_DIR)

# ===== Dataset split: train (70%), val (15%), test (15%) =====
train_ds = keras.utils.image_dataset_from_directory(
    data_dir, labels="inferred", label_mode="int",
    image_size=IMG_SIZE, batch_size=BATCH_SIZE,
    validation_split=0.15, subset="training", seed=SEED, shuffle=True
)

valtest_ds = keras.utils.image_dataset_from_directory(
    data_dir, labels="inferred", label_mode="int",
    image_size=IMG_SIZE, batch_size=BATCH_SIZE,
    validation_split=0.15, subset="validation", seed=SEED, shuffle=False
)

# Split val/test 50/50 from the validation subset
val_batches = int(0.5 * tf.data.experimental.cardinality(valtest_ds).numpy())
val_ds = valtest_ds.take(val_batches)
test_ds = valtest_ds.skip(val_batches)

class_names = train_ds.class_names
num_classes = len(class_names)
print("Classes:", class_names)

# ===== Prefetch for performance (same as second snippet) =====
AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_ds.shuffle(1000).prefetch(AUTOTUNE)
val_ds   = val_ds.prefetch(AUTOTUNE)
test_ds  = test_ds.prefetch(AUTOTUNE)

# ===== Data augmentation (same as second snippet) =====
data_augmentation = keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.05),
    layers.RandomZoom(0.1),
])

# ===== Normalization (same as second snippet) =====
# NOTE: We use simple Rescaling(1./255) for ALL models, including transfer learning.
# This matches the second snippet; it's fine for your experiments.
def normalize_block(x):
    return layers.Rescaling(1./255)(x)

# =========================================
# Classifier head (shared for TL models)
# =========================================
def classifier_head(x, num_classes, dropout=0.3):
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dropout(dropout)(x)
    return layers.Dense(num_classes, activation="softmax")(x)

# =========================================
# 1) Baseline CNN (from scratch)
#   - keep architecture flexible, but preprocessing matches second snippet
# =========================================
def build_baseline_cnn(input_shape, num_classes):
    inputs = keras.Input(shape=input_shape)
    x = data_augmentation(inputs)
    x = normalize_block(x)

    # Three conv blocks, two convs per block (as in your first snippet's baseline)
    for filters in [32, 64, 128]:
        x = layers.Conv2D(filters, 3, padding="same", activation="relu")(x)
        x = layers.Conv2D(filters, 3, padding="same", activation="relu")(x)
        x = layers.MaxPooling2D()(x)
        x = layers.Dropout(0.25)(x)

    x = layers.Flatten()(x)
    x = layers.Dense(256, activation="relu")(x)
    x = layers.Dropout(0.5)(x)
    outputs = layers.Dense(num_classes, activation="softmax")(x)
    return keras.Model(inputs, outputs, name="baseline_cnn")

# =========================================
# 2) ResNet50 (prebuilt)
# =========================================
from tensorflow.keras.applications import ResNet50
def build_resnet50(input_shape, num_classes, weights="imagenet", train_base=False):
    inputs = keras.Input(shape=input_shape)
    x = data_augmentation(inputs)
    x = normalize_block(x)
    base = ResNet50(include_top=False, weights=weights, input_tensor=x)
    base.trainable = train_base
    outputs = classifier_head(base.output, num_classes, dropout=0.3)
    return keras.Model(inputs, outputs, name="resnet50")

# =========================================
# 3) VGG16 (prebuilt)
# =========================================
from tensorflow.keras.applications import VGG16
def build_vgg16(input_shape, num_classes, weights="imagenet", train_base=False):
    inputs = keras.Input(shape=input_shape)
    x = data_augmentation(inputs)
    x = normalize_block(x)
    base = VGG16(include_top=False, weights=weights, input_tensor=x)
    base.trainable = train_base
    outputs = classifier_head(base.output, num_classes, dropout=0.3)
    return keras.Model(inputs, outputs, name="vgg16")

# =========================================
# 4) InceptionV3 (prebuilt)
# =========================================
from tensorflow.keras.applications import InceptionV3
def build_inceptionv3(input_shape, num_classes, weights="imagenet", train_base=False):
    inputs = keras.Input(shape=input_shape)
    x = data_augmentation(inputs)
    x = normalize_block(x)
    base = InceptionV3(include_top=False, weights=weights, input_tensor=x)
    base.trainable = train_base
    outputs = classifier_head(base.output, num_classes, dropout=0.3)
    return keras.Model(inputs, outputs, name="inceptionv3")

# =========================================
# 5) EfficientNetB0 (prebuilt)
# =========================================
from tensorflow.keras.applications import EfficientNetB0
def build_efficientnetb0(input_shape, num_classes, weights="imagenet", train_base=False):
    inputs = keras.Input(shape=input_shape)
    x = data_augmentation(inputs)
    x = normalize_block(x)
    base = EfficientNetB0(include_top=False, weights=weights, input_tensor=x)
    base.trainable = train_base
    outputs = classifier_head(base.output, num_classes, dropout=0.3)
    return keras.Model(inputs, outputs, name="efficientnetb0")

# =========================================
# Pick a model to train (same interface)
# =========================================
MODEL_NAME = "resnet"   # one of: "cnn", "resnet", "vgg", "inception", "efficientnet"
WEIGHTS = "imagenet"          # "imagenet" or None
FINE_TUNE_BASE = False        # set True to unfreeze backbone later

input_shape = (*IMG_SIZE, 3)

if MODEL_NAME == "cnn":
    model = build_baseline_cnn(input_shape, num_classes)
elif MODEL_NAME == "resnet":
    model = build_resnet50(input_shape, num_classes, weights=WEIGHTS, train_base=FINE_TUNE_BASE)
elif MODEL_NAME == "vgg":
    model = build_vgg16(input_shape, num_classes, weights=WEIGHTS, train_base=FINE_TUNE_BASE)
elif MODEL_NAME == "inception":
    model = build_inceptionv3(input_shape, num_classes, weights=WEIGHTS, train_base=FINE_TUNE_BASE)
elif MODEL_NAME == "efficientnet":
    model = build_efficientnetb0(input_shape, num_classes, weights=WEIGHTS, train_base=FINE_TUNE_BASE)
else:
    raise ValueError("Unknown MODEL_NAME")

# =========================================
# Compile & Train
# =========================================
model.compile(
    optimizer=keras.optimizers.Adam(1e-3),
    loss=keras.losses.SparseCategoricalCrossentropy(),
    metrics=["accuracy"]
)

callbacks = [
    keras.callbacks.EarlyStopping(patience=4, restore_best_weights=True, monitor="val_accuracy"),
]

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=20,
    callbacks=callbacks
)

# ===== Evaluate on the held-out TEST set (matches second snippet practice) =====
test_loss, test_acc = model.evaluate(test_ds, verbose=0)
print(f"{model.name}  |  Test acc: {test_acc:.4f}  |  Test loss: {test_loss:.4f}")

Found 10000 files belonging to 2 classes.
Using 8500 files for training.
Found 10000 files belonging to 2 classes.
Using 1500 files for validation.
Classes: ['breast_benign', 'breast_malignant']
Epoch 1/20
1063/1063 [==============================] - 33s 20ms/step - loss: 0.6374 - accuracy: 0.6521 - val_loss: 0.8576 - val_accuracy: 0.4322
Epoch 2/20
1063/1063 [==============================] - 27s 19ms/step - loss: 0.6083 - accuracy: 0.6756 - val_loss: 0.6389 - val_accuracy: 0.6622
Epoch 3/20
1063/1063 [==============================] - 26s 18ms/step - loss: 0.5994 - accuracy: 0.6909 - val_loss: 0.7078 - val_accuracy: 0.5891
Epoch 4/20
1063/1063 [==============================] - 28s 19ms/step - loss: 0.5980 - accuracy: 0.6971 - val_loss: 0.7396 - val_accuracy: 0.5745
Epoch 5/20
1063/1063 [==============================] - 26s 18ms/step - loss: 0.5855 - accuracy: 0.7018 - val_loss: 0.6969 - val_accuracy: 0.5971
Epoch 6/20
1063/1063 [==============================] - 26s 18ms/step - los

In [3]:

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import pathlib

# ---- Paths & data ----
DATA_DIR = "/home/jovyan/.cache/kagglehub/datasets/obulisainaren/multi-cancer/versions/3/Multi Cancer/Multi Cancer/Breast Cancer"
IMG_SIZE = (256, 256)   
BATCH_SIZE = 8
SEED = 1337

data_dir = pathlib.Path(DATA_DIR)

# ===== Dataset split: train (70%), val (15%), test (15%) =====
train_ds = keras.utils.image_dataset_from_directory(
    data_dir, labels="inferred", label_mode="int",
    image_size=IMG_SIZE, batch_size=BATCH_SIZE,
    validation_split=0.15, subset="training", seed=SEED, shuffle=True
)

valtest_ds = keras.utils.image_dataset_from_directory(
    data_dir, labels="inferred", label_mode="int",
    image_size=IMG_SIZE, batch_size=BATCH_SIZE,
    validation_split=0.15, subset="validation", seed=SEED, shuffle=False
)

# Split val/test 50/50 from the validation subset
val_batches = int(0.5 * tf.data.experimental.cardinality(valtest_ds).numpy())
val_ds = valtest_ds.take(val_batches)
test_ds = valtest_ds.skip(val_batches)

class_names = train_ds.class_names
num_classes = len(class_names)
print("Classes:", class_names)

# ===== Prefetch for performance (same as second snippet) =====
AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_ds.shuffle(1000).prefetch(AUTOTUNE)
val_ds   = val_ds.prefetch(AUTOTUNE)
test_ds  = test_ds.prefetch(AUTOTUNE)

# ===== Data augmentation (same as second snippet) =====
data_augmentation = keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.05),
    layers.RandomZoom(0.1),
])

# ===== Normalization (same as second snippet) =====
# NOTE: We use simple Rescaling(1./255) for ALL models, including transfer learning.
# This matches the second snippet; it's fine for your experiments.
def normalize_block(x):
    return layers.Rescaling(1./255)(x)

# =========================================
# Classifier head (shared for TL models)
# =========================================
def classifier_head(x, num_classes, dropout=0.3):
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dropout(dropout)(x)
    return layers.Dense(num_classes, activation="softmax")(x)

# =========================================
# 1) Baseline CNN (from scratch)
#   - keep architecture flexible, but preprocessing matches second snippet
# =========================================
def build_baseline_cnn(input_shape, num_classes):
    inputs = keras.Input(shape=input_shape)
    x = data_augmentation(inputs)
    x = normalize_block(x)

    # Three conv blocks, two convs per block (as in your first snippet's baseline)
    for filters in [32, 64, 128]:
        x = layers.Conv2D(filters, 3, padding="same", activation="relu")(x)
        x = layers.Conv2D(filters, 3, padding="same", activation="relu")(x)
        x = layers.MaxPooling2D()(x)
        x = layers.Dropout(0.25)(x)

    x = layers.Flatten()(x)
    x = layers.Dense(256, activation="relu")(x)
    x = layers.Dropout(0.5)(x)
    outputs = layers.Dense(num_classes, activation="softmax")(x)
    return keras.Model(inputs, outputs, name="baseline_cnn")

# =========================================
# 2) ResNet50 (prebuilt)
# =========================================
from tensorflow.keras.applications import ResNet50
def build_resnet50(input_shape, num_classes, weights="imagenet", train_base=False):
    inputs = keras.Input(shape=input_shape)
    x = data_augmentation(inputs)
    x = normalize_block(x)
    base = ResNet50(include_top=False, weights=weights, input_tensor=x)
    base.trainable = train_base
    outputs = classifier_head(base.output, num_classes, dropout=0.3)
    return keras.Model(inputs, outputs, name="resnet50")

# =========================================
# 3) VGG16 (prebuilt)
# =========================================
from tensorflow.keras.applications import VGG16
def build_vgg16(input_shape, num_classes, weights="imagenet", train_base=False):
    inputs = keras.Input(shape=input_shape)
    x = data_augmentation(inputs)
    x = normalize_block(x)
    base = VGG16(include_top=False, weights=weights, input_tensor=x)
    base.trainable = train_base
    outputs = classifier_head(base.output, num_classes, dropout=0.3)
    return keras.Model(inputs, outputs, name="vgg16")

# =========================================
# 4) InceptionV3 (prebuilt)
# =========================================
from tensorflow.keras.applications import InceptionV3
def build_inceptionv3(input_shape, num_classes, weights="imagenet", train_base=False):
    inputs = keras.Input(shape=input_shape)
    x = data_augmentation(inputs)
    x = normalize_block(x)
    base = InceptionV3(include_top=False, weights=weights, input_tensor=x)
    base.trainable = train_base
    outputs = classifier_head(base.output, num_classes, dropout=0.3)
    return keras.Model(inputs, outputs, name="inceptionv3")

# =========================================
# 5) EfficientNetB0 (prebuilt)
# =========================================
from tensorflow.keras.applications import EfficientNetB0
def build_efficientnetb0(input_shape, num_classes, weights="imagenet", train_base=False):
    inputs = keras.Input(shape=input_shape)
    x = data_augmentation(inputs)
    x = normalize_block(x)
    base = EfficientNetB0(include_top=False, weights=weights, input_tensor=x)
    base.trainable = train_base
    outputs = classifier_head(base.output, num_classes, dropout=0.3)
    return keras.Model(inputs, outputs, name="efficientnetb0")

# =========================================
# Pick a model to train (same interface)
# =========================================
MODEL_NAME = "vgg"   # one of: "cnn", "resnet", "vgg", "inception", "efficientnet"
WEIGHTS = "imagenet"          # "imagenet" or None
FINE_TUNE_BASE = False        # set True to unfreeze backbone later

input_shape = (*IMG_SIZE, 3)

if MODEL_NAME == "cnn":
    model = build_baseline_cnn(input_shape, num_classes)
elif MODEL_NAME == "resnet":
    model = build_resnet50(input_shape, num_classes, weights=WEIGHTS, train_base=FINE_TUNE_BASE)
elif MODEL_NAME == "vgg":
    model = build_vgg16(input_shape, num_classes, weights=WEIGHTS, train_base=FINE_TUNE_BASE)
elif MODEL_NAME == "inception":
    model = build_inceptionv3(input_shape, num_classes, weights=WEIGHTS, train_base=FINE_TUNE_BASE)
elif MODEL_NAME == "efficientnet":
    model = build_efficientnetb0(input_shape, num_classes, weights=WEIGHTS, train_base=FINE_TUNE_BASE)
else:
    raise ValueError("Unknown MODEL_NAME")

# =========================================
# Compile & Train
# =========================================
model.compile(
    optimizer=keras.optimizers.Adam(1e-3),
    loss=keras.losses.SparseCategoricalCrossentropy(),
    metrics=["accuracy"]
)

callbacks = [
    keras.callbacks.EarlyStopping(patience=4, restore_best_weights=True, monitor="val_accuracy"),
]

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=20,
    callbacks=callbacks
)

# ===== Evaluate on the held-out TEST set (matches second snippet practice) =====
test_loss, test_acc = model.evaluate(test_ds, verbose=0)
print(f"{model.name}  |  Test acc: {test_acc:.4f}  |  Test loss: {test_loss:.4f}")

Found 10000 files belonging to 2 classes.
Using 8500 files for training.
Found 10000 files belonging to 2 classes.
Using 1500 files for validation.
Classes: ['breast_benign', 'breast_malignant']
Epoch 1/20
1063/1063 [==============================] - 34s 23ms/step - loss: 0.4289 - accuracy: 0.8145 - val_loss: 0.3291 - val_accuracy: 0.8856
Epoch 2/20
1063/1063 [==============================] - 30s 22ms/step - loss: 0.3179 - accuracy: 0.8725 - val_loss: 0.2308 - val_accuracy: 0.9255
Epoch 3/20
1063/1063 [==============================] - 30s 22ms/step - loss: 0.2816 - accuracy: 0.8893 - val_loss: 0.2431 - val_accuracy: 0.9149
Epoch 4/20
1063/1063 [==============================] - 29s 21ms/step - loss: 0.2667 - accuracy: 0.8940 - val_loss: 0.1359 - val_accuracy: 0.9614
Epoch 5/20
1063/1063 [==============================] - 30s 21ms/step - loss: 0.2531 - accuracy: 0.8996 - val_loss: 0.2557 - val_accuracy: 0.9229
Epoch 6/20
1063/1063 [==============================] - 30s 21ms/step - los

In [4]:

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import pathlib

# ---- Paths & data ----
DATA_DIR = "/home/jovyan/.cache/kagglehub/datasets/obulisainaren/multi-cancer/versions/3/Multi Cancer/Multi Cancer/Breast Cancer"
IMG_SIZE = (256, 256)   
BATCH_SIZE = 8
SEED = 1337

data_dir = pathlib.Path(DATA_DIR)

# ===== Dataset split: train (70%), val (15%), test (15%) =====
train_ds = keras.utils.image_dataset_from_directory(
    data_dir, labels="inferred", label_mode="int",
    image_size=IMG_SIZE, batch_size=BATCH_SIZE,
    validation_split=0.15, subset="training", seed=SEED, shuffle=True
)

valtest_ds = keras.utils.image_dataset_from_directory(
    data_dir, labels="inferred", label_mode="int",
    image_size=IMG_SIZE, batch_size=BATCH_SIZE,
    validation_split=0.15, subset="validation", seed=SEED, shuffle=False
)

# Split val/test 50/50 from the validation subset
val_batches = int(0.5 * tf.data.experimental.cardinality(valtest_ds).numpy())
val_ds = valtest_ds.take(val_batches)
test_ds = valtest_ds.skip(val_batches)

class_names = train_ds.class_names
num_classes = len(class_names)
print("Classes:", class_names)

# ===== Prefetch for performance (same as second snippet) =====
AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_ds.shuffle(1000).prefetch(AUTOTUNE)
val_ds   = val_ds.prefetch(AUTOTUNE)
test_ds  = test_ds.prefetch(AUTOTUNE)

# ===== Data augmentation (same as second snippet) =====
data_augmentation = keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.05),
    layers.RandomZoom(0.1),
])

# ===== Normalization (same as second snippet) =====
# NOTE: We use simple Rescaling(1./255) for ALL models, including transfer learning.
# This matches the second snippet; it's fine for your experiments.
def normalize_block(x):
    return layers.Rescaling(1./255)(x)

# =========================================
# Classifier head (shared for TL models)
# =========================================
def classifier_head(x, num_classes, dropout=0.3):
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dropout(dropout)(x)
    return layers.Dense(num_classes, activation="softmax")(x)

# =========================================
# 1) Baseline CNN (from scratch)
#   - keep architecture flexible, but preprocessing matches second snippet
# =========================================
def build_baseline_cnn(input_shape, num_classes):
    inputs = keras.Input(shape=input_shape)
    x = data_augmentation(inputs)
    x = normalize_block(x)

    # Three conv blocks, two convs per block (as in your first snippet's baseline)
    for filters in [32, 64, 128]:
        x = layers.Conv2D(filters, 3, padding="same", activation="relu")(x)
        x = layers.Conv2D(filters, 3, padding="same", activation="relu")(x)
        x = layers.MaxPooling2D()(x)
        x = layers.Dropout(0.25)(x)

    x = layers.Flatten()(x)
    x = layers.Dense(256, activation="relu")(x)
    x = layers.Dropout(0.5)(x)
    outputs = layers.Dense(num_classes, activation="softmax")(x)
    return keras.Model(inputs, outputs, name="baseline_cnn")

# =========================================
# 2) ResNet50 (prebuilt)
# =========================================
from tensorflow.keras.applications import ResNet50
def build_resnet50(input_shape, num_classes, weights="imagenet", train_base=False):
    inputs = keras.Input(shape=input_shape)
    x = data_augmentation(inputs)
    x = normalize_block(x)
    base = ResNet50(include_top=False, weights=weights, input_tensor=x)
    base.trainable = train_base
    outputs = classifier_head(base.output, num_classes, dropout=0.3)
    return keras.Model(inputs, outputs, name="resnet50")

# =========================================
# 3) VGG16 (prebuilt)
# =========================================
from tensorflow.keras.applications import VGG16
def build_vgg16(input_shape, num_classes, weights="imagenet", train_base=False):
    inputs = keras.Input(shape=input_shape)
    x = data_augmentation(inputs)
    x = normalize_block(x)
    base = VGG16(include_top=False, weights=weights, input_tensor=x)
    base.trainable = train_base
    outputs = classifier_head(base.output, num_classes, dropout=0.3)
    return keras.Model(inputs, outputs, name="vgg16")

# =========================================
# 4) InceptionV3 (prebuilt)
# =========================================
from tensorflow.keras.applications import InceptionV3
def build_inceptionv3(input_shape, num_classes, weights="imagenet", train_base=False):
    inputs = keras.Input(shape=input_shape)
    x = data_augmentation(inputs)
    x = normalize_block(x)
    base = InceptionV3(include_top=False, weights=weights, input_tensor=x)
    base.trainable = train_base
    outputs = classifier_head(base.output, num_classes, dropout=0.3)
    return keras.Model(inputs, outputs, name="inceptionv3")

# =========================================
# 5) EfficientNetB0 (prebuilt)
# =========================================
from tensorflow.keras.applications import EfficientNetB0
def build_efficientnetb0(input_shape, num_classes, weights="imagenet", train_base=False):
    inputs = keras.Input(shape=input_shape)
    x = data_augmentation(inputs)
    x = normalize_block(x)
    base = EfficientNetB0(include_top=False, weights=weights, input_tensor=x)
    base.trainable = train_base
    outputs = classifier_head(base.output, num_classes, dropout=0.3)
    return keras.Model(inputs, outputs, name="efficientnetb0")

# =========================================
# Pick a model to train (same interface)
# =========================================
MODEL_NAME = "inception"   # one of: "cnn", "resnet", "vgg", "inception", "efficientnet"
WEIGHTS = "imagenet"          # "imagenet" or None
FINE_TUNE_BASE = False        # set True to unfreeze backbone later

input_shape = (*IMG_SIZE, 3)

if MODEL_NAME == "cnn":
    model = build_baseline_cnn(input_shape, num_classes)
elif MODEL_NAME == "resnet":
    model = build_resnet50(input_shape, num_classes, weights=WEIGHTS, train_base=FINE_TUNE_BASE)
elif MODEL_NAME == "vgg":
    model = build_vgg16(input_shape, num_classes, weights=WEIGHTS, train_base=FINE_TUNE_BASE)
elif MODEL_NAME == "inception":
    model = build_inceptionv3(input_shape, num_classes, weights=WEIGHTS, train_base=FINE_TUNE_BASE)
elif MODEL_NAME == "efficientnet":
    model = build_efficientnetb0(input_shape, num_classes, weights=WEIGHTS, train_base=FINE_TUNE_BASE)
else:
    raise ValueError("Unknown MODEL_NAME")

# =========================================
# Compile & Train
# =========================================
model.compile(
    optimizer=keras.optimizers.Adam(1e-3),
    loss=keras.losses.SparseCategoricalCrossentropy(),
    metrics=["accuracy"]
)

callbacks = [
    keras.callbacks.EarlyStopping(patience=4, restore_best_weights=True, monitor="val_accuracy"),
]

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=20,
    callbacks=callbacks
)

# ===== Evaluate on the held-out TEST set (matches second snippet practice) =====
test_loss, test_acc = model.evaluate(test_ds, verbose=0)
print(f"{model.name}  |  Test acc: {test_acc:.4f}  |  Test loss: {test_loss:.4f}")

Found 10000 files belonging to 2 classes.
Using 8500 files for training.
Found 10000 files belonging to 2 classes.
Using 1500 files for validation.
Classes: ['breast_benign', 'breast_malignant']
Epoch 1/20
1063/1063 [==============================] - 33s 19ms/step - loss: 0.2966 - accuracy: 0.8735 - val_loss: 0.2593 - val_accuracy: 0.8856
Epoch 2/20
1063/1063 [==============================] - 24s 16ms/step - loss: 0.2326 - accuracy: 0.9084 - val_loss: 0.2102 - val_accuracy: 0.9149
Epoch 3/20
1063/1063 [==============================] - 24s 16ms/step - loss: 0.2296 - accuracy: 0.9071 - val_loss: 0.1185 - val_accuracy: 0.9614
Epoch 4/20
1063/1063 [==============================] - 24s 16ms/step - loss: 0.2290 - accuracy: 0.9082 - val_loss: 0.2553 - val_accuracy: 0.8870
Epoch 5/20
1063/1063 [==============================] - 25s 17ms/step - loss: 0.2381 - accuracy: 0.9085 - val_loss: 0.0564 - val_accuracy: 0.9761
Epoch 6/20
1063/1063 [==============================] - 24s 16ms/step - los

In [5]:

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import pathlib

# ---- Paths & data ----
DATA_DIR = "/home/jovyan/.cache/kagglehub/datasets/obulisainaren/multi-cancer/versions/3/Multi Cancer/Multi Cancer/Breast Cancer"
IMG_SIZE = (256, 256)   
BATCH_SIZE = 8
SEED = 1337

data_dir = pathlib.Path(DATA_DIR)

# ===== Dataset split: train (70%), val (15%), test (15%) =====
train_ds = keras.utils.image_dataset_from_directory(
    data_dir, labels="inferred", label_mode="int",
    image_size=IMG_SIZE, batch_size=BATCH_SIZE,
    validation_split=0.15, subset="training", seed=SEED, shuffle=True
)

valtest_ds = keras.utils.image_dataset_from_directory(
    data_dir, labels="inferred", label_mode="int",
    image_size=IMG_SIZE, batch_size=BATCH_SIZE,
    validation_split=0.15, subset="validation", seed=SEED, shuffle=False
)

# Split val/test 50/50 from the validation subset
val_batches = int(0.5 * tf.data.experimental.cardinality(valtest_ds).numpy())
val_ds = valtest_ds.take(val_batches)
test_ds = valtest_ds.skip(val_batches)

class_names = train_ds.class_names
num_classes = len(class_names)
print("Classes:", class_names)

# ===== Prefetch for performance (same as second snippet) =====
AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_ds.shuffle(1000).prefetch(AUTOTUNE)
val_ds   = val_ds.prefetch(AUTOTUNE)
test_ds  = test_ds.prefetch(AUTOTUNE)

# ===== Data augmentation (same as second snippet) =====
data_augmentation = keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.05),
    layers.RandomZoom(0.1),
])

# ===== Normalization (same as second snippet) =====
# NOTE: We use simple Rescaling(1./255) for ALL models, including transfer learning.
# This matches the second snippet; it's fine for your experiments.
def normalize_block(x):
    return layers.Rescaling(1./255)(x)

# =========================================
# Classifier head (shared for TL models)
# =========================================
def classifier_head(x, num_classes, dropout=0.3):
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dropout(dropout)(x)
    return layers.Dense(num_classes, activation="softmax")(x)

# =========================================
# 1) Baseline CNN (from scratch)
#   - keep architecture flexible, but preprocessing matches second snippet
# =========================================
def build_baseline_cnn(input_shape, num_classes):
    inputs = keras.Input(shape=input_shape)
    x = data_augmentation(inputs)
    x = normalize_block(x)

    # Three conv blocks, two convs per block (as in your first snippet's baseline)
    for filters in [32, 64, 128]:
        x = layers.Conv2D(filters, 3, padding="same", activation="relu")(x)
        x = layers.Conv2D(filters, 3, padding="same", activation="relu")(x)
        x = layers.MaxPooling2D()(x)
        x = layers.Dropout(0.25)(x)

    x = layers.Flatten()(x)
    x = layers.Dense(256, activation="relu")(x)
    x = layers.Dropout(0.5)(x)
    outputs = layers.Dense(num_classes, activation="softmax")(x)
    return keras.Model(inputs, outputs, name="baseline_cnn")

# =========================================
# 2) ResNet50 (prebuilt)
# =========================================
from tensorflow.keras.applications import ResNet50
def build_resnet50(input_shape, num_classes, weights="imagenet", train_base=False):
    inputs = keras.Input(shape=input_shape)
    x = data_augmentation(inputs)
    x = normalize_block(x)
    base = ResNet50(include_top=False, weights=weights, input_tensor=x)
    base.trainable = train_base
    outputs = classifier_head(base.output, num_classes, dropout=0.3)
    return keras.Model(inputs, outputs, name="resnet50")

# =========================================
# 3) VGG16 (prebuilt)
# =========================================
from tensorflow.keras.applications import VGG16
def build_vgg16(input_shape, num_classes, weights="imagenet", train_base=False):
    inputs = keras.Input(shape=input_shape)
    x = data_augmentation(inputs)
    x = normalize_block(x)
    base = VGG16(include_top=False, weights=weights, input_tensor=x)
    base.trainable = train_base
    outputs = classifier_head(base.output, num_classes, dropout=0.3)
    return keras.Model(inputs, outputs, name="vgg16")

# =========================================
# 4) InceptionV3 (prebuilt)
# =========================================
from tensorflow.keras.applications import InceptionV3
def build_inceptionv3(input_shape, num_classes, weights="imagenet", train_base=False):
    inputs = keras.Input(shape=input_shape)
    x = data_augmentation(inputs)
    x = normalize_block(x)
    base = InceptionV3(include_top=False, weights=weights, input_tensor=x)
    base.trainable = train_base
    outputs = classifier_head(base.output, num_classes, dropout=0.3)
    return keras.Model(inputs, outputs, name="inceptionv3")

# =========================================
# 5) EfficientNetB0 (prebuilt)
# =========================================
from tensorflow.keras.applications import EfficientNetB0
def build_efficientnetb0(input_shape, num_classes, weights="imagenet", train_base=False):
    inputs = keras.Input(shape=input_shape)
    x = data_augmentation(inputs)
    x = normalize_block(x)
    base = EfficientNetB0(include_top=False, weights=weights, input_tensor=x)
    base.trainable = train_base
    outputs = classifier_head(base.output, num_classes, dropout=0.3)
    return keras.Model(inputs, outputs, name="efficientnetb0")

# =========================================
# Pick a model to train (same interface)
# =========================================
MODEL_NAME = "efficientnet"   # one of: "cnn", "resnet", "vgg", "inception", "efficientnet"
WEIGHTS = "imagenet"          # "imagenet" or None
FINE_TUNE_BASE = False        # set True to unfreeze backbone later

input_shape = (*IMG_SIZE, 3)

if MODEL_NAME == "cnn":
    model = build_baseline_cnn(input_shape, num_classes)
elif MODEL_NAME == "resnet":
    model = build_resnet50(input_shape, num_classes, weights=WEIGHTS, train_base=FINE_TUNE_BASE)
elif MODEL_NAME == "vgg":
    model = build_vgg16(input_shape, num_classes, weights=WEIGHTS, train_base=FINE_TUNE_BASE)
elif MODEL_NAME == "inception":
    model = build_inceptionv3(input_shape, num_classes, weights=WEIGHTS, train_base=FINE_TUNE_BASE)
elif MODEL_NAME == "efficientnet":
    model = build_efficientnetb0(input_shape, num_classes, weights=WEIGHTS, train_base=FINE_TUNE_BASE)
else:
    raise ValueError("Unknown MODEL_NAME")

# =========================================
# Compile & Train
# =========================================
model.compile(
    optimizer=keras.optimizers.Adam(1e-3),
    loss=keras.losses.SparseCategoricalCrossentropy(),
    metrics=["accuracy"]
)

callbacks = [
    keras.callbacks.EarlyStopping(patience=4, restore_best_weights=True, monitor="val_accuracy"),
]

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=20,
    callbacks=callbacks
)

# ===== Evaluate on the held-out TEST set (matches second snippet practice) =====
test_loss, test_acc = model.evaluate(test_ds, verbose=0)
print(f"{model.name}  |  Test acc: {test_acc:.4f}  |  Test loss: {test_loss:.4f}")

Found 10000 files belonging to 2 classes.
Using 8500 files for training.
Found 10000 files belonging to 2 classes.
Using 1500 files for validation.
Classes: ['breast_benign', 'breast_malignant']
Epoch 1/20
1063/1063 [==============================] - 32s 17ms/step - loss: 0.7157 - accuracy: 0.4974 - val_loss: 0.5321 - val_accuracy: 1.0000
Epoch 2/20
1063/1063 [==============================] - 22s 14ms/step - loss: 0.7092 - accuracy: 0.5171 - val_loss: 0.5821 - val_accuracy: 1.0000
Epoch 3/20
1063/1063 [==============================] - 22s 14ms/step - loss: 0.7050 - accuracy: 0.5234 - val_loss: 0.7162 - val_accuracy: 0.0346
Epoch 4/20
1063/1063 [==============================] - 24s 16ms/step - loss: 0.7068 - accuracy: 0.5164 - val_loss: 0.6393 - val_accuracy: 0.9947
Epoch 5/20
1063/1063 [==============================] - 24s 16ms/step - loss: 0.7047 - accuracy: 0.5227 - val_loss: 0.4331 - val_accuracy: 1.0000
efficientnetb0  |  Test acc: 1.0000  |  Test loss: 0.5351
